<a href="https://colab.research.google.com/github/vikassingh0593/bytemaster_stocks/blob/dev/web_scrapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Update package lists silently
!apt-get update -qq > /dev/null

# Install OpenJDK 11 (required for Spark)
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# Download Spark 3.1.1 with Hadoop 3.2
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz

# Extract the downloaded Spark archive
!tar xf spark-3.1.1-bin-hadoop3.2.tgz

# Install the 'findspark' library for easy integration
!pip install -q findspark

# Set environment variables for Java and Spark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"  # Path to OpenJDK 11
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"  # Path to Spark

# Initialize findspark and PySpark
import findspark
findspark.init()

from pyspark.sql import SparkSession
# Create a Spark session
spark = SparkSession.builder.master("local[*]").appName("MySparkApp").getOrCreate()

# Verify the Spark session
print("Spark version:", spark.version)  # Print the Spark version to confirm setup


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Spark version: 3.1.1


In [10]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pyspark.sql.functions import col, when, last, monotonically_increasing_id, lag, lead, coalesce, lit
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql import DataFrame

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lower
from pyspark.sql import functions as F

In [11]:
# Install PySpark and yfinance
!pip install pyspark yfinance

import yfinance as yf
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

import os
from google.colab import drive
!pip install yfinance
# !pip install --upgrade numpy
from yfinance import Ticker
spark.conf.set("spark.sql.debug.maxToStringFields", 1000) # Or a higher value as needed

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [33]:
def rd_fn(Number, Qtr, Yer):

  url = f"https://www.bseindia.com/corporates/shpPublicShareholder.aspx?scripcd={Number}&qtrid=121.00&QtrName={Qtr}%20{Yer}"

  # Define headers to mimic a real browser request
  headers = {
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
  }

  # Define cookies if required
  cookies = {
      "cookie_name": "cookie_value"
  }

  # Send a GET request to the URL with headers and cookies
  response = requests.get(url, headers=headers, cookies=cookies)

  soup = BeautifulSoup(response.content, "html.parser")

  rows = soup.find_all('tr')

  for td in rows[7].find_all('td'):
    title = td.get_text().strip()

  data_lst = []
  for i in range(17, len(rows)):
    try:
      text_list = [td.get_text().strip() for td in rows[i].find_all('td')[1:8]]
      if text_list[0]=='B1) Institutions':
        append_flag = True
      if append_flag:
        data_lst.append(text_list)
      i+=1
    except IndexError:
      append_flag = False

  # for i in data_lst:
  #   print(len(i), i)

  columns = ['CategoryNameoftheShareholders', 'NoOfShareholder', 'NoOfFullyPaidShares', 'Blank_1', 'Blank_2', 'TotalNoSharesHeld', 'ShareholdingPerc']
  pandas_df = pd.DataFrame(data_lst, columns=columns)

  # Replace null values and empty spaces with 0
  # pandas_df.replace(to_replace=['', ' ', None, pd.NA], value=0, inplace=True)

  # Convert data types
  pandas_df['NoOfShareholder'] = pandas_df['NoOfShareholder'].astype(int)
  pandas_df['ShareholdingPerc'] = pandas_df['ShareholdingPerc'].astype(float)

  # Filter rows where NoOfShareholder is equal to 1
  pandas_df = pandas_df[pandas_df['NoOfShareholder'] == 1]

  # Drop the original Blank_1 and Blank_2 columns
  pandas_df.drop(columns=['Blank_1', 'Blank_2', 'NoOfShareholder', 'TotalNoSharesHeld'], inplace=True)
  pandas_df['Number'] = Number
  pandas_df['Title'] = title
  pandas_df['Qtr'] = Qtr
  pandas_df['Year'] = Yer

  return pandas_df

In [34]:
Number = 500033
Qtr = "March"
Yer = "2024"
df = rd_fn(Number, Qtr, Yer)
df

,CategoryNameoftheShareholders,NoOfFullyPaidShares,ShareholdingPerc,Number,Title,Qtr,Year
2,Mutual Funds/,67000,0.51,500033,FORCE MOTORS LTD.-$,March,2024
9,Point Break Capital LLP,140679,1.07,500033,FORCE MOTORS LTD.-$,March,2024
13,Directors and their relatives (excluding indep...,1189,0.01,500033,FORCE MOTORS LTD.-$,March,2024
14,Investor Education and Protection Fund (IEPF),53436,0.41,500033,FORCE MOTORS LTD.-$,March,2024
17,Vanaja Sundar Iyer,190000,1.44,500033,FORCE MOTORS LTD.-$,March,2024
19,Manohar Devabhaktuni,158035,1.20,500033,FORCE MOTORS LTD.-$,March,2024
23,Unclaimed or Suspense or Escrow Account,200,0.00,500033,FORCE MOTORS LTD.-$,March,2024


In [ ]:
################################################################# ROUGH #################################################################

In [ ]:
# k=0
# i=0
# data_lst = []
# while k==0:
#   try:
#     text_list = [td.get_text().strip() for td in rows[i+17].find_all('td')[1:8]]
#     if text_list[0]=='B1) Institutions':
#       append_flag = True
#     if append_flag:
#       data_lst.append(text_list)
#     i+=1
#   except:
#     append_flag = False
#     k+=1

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

# Initialize SparkSession
spark = SparkSession.builder.appName("Empty DataFrame Example").getOrCreate()

# Define schema
schema = StructType([
    StructField("CategoryNameoftheShareholders", StringType(), True),
    StructField("NoOfShareholder", StringType(), True),
    StructField("NoOfFullyPaidShares", StringType(), True),
    StructField("Blank_1", StringType(), True),
    StructField("Blank_2", StringType(), True),
    StructField("TotalNoSharesHeld", StringType(), True),
    StructField("ShareholdingPerc", StringType(), True)
])

# Create an empty DataFrame
empty_df = spark.createDataFrame([], schema)

# Show the empty DataFrame
empty_df.show()


In [ ]:
# url = f"https://www.bseindia.com/corporates/shpPublicShareholder.aspx?scripcd=500033&qtrid=121.00&QtrName=March%202024"
Number = 500033
Qtr = "March"
Yer = "2024"
url = f"https://www.bseindia.com/corporates/shpPublicShareholder.aspx?scripcd={Number}&qtrid=121.00&QtrName={Qtr}%20{Yer}"

# Define headers to mimic a real browser request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

# Define cookies if required
cookies = {
    "cookie_name": "cookie_value"
}

# Send a GET request to the URL with headers and cookies
response = requests.get(url, headers=headers, cookies=cookies)

soup = BeautifulSoup(response.content, "html.parser")

rows = soup.find_all('tr')

for td in rows[7].find_all('td'):
  title = td.get_text().strip()

data_lst = []
for i in range(17, len(rows)):
  try:
    text_list = [td.get_text().strip() for td in rows[i].find_all('td')[1:8]]
    if text_list[0]=='B1) Institutions':
      append_flag = True
    if append_flag:
      data_lst.append(text_list)
    i+=1
  except IndexError:
    append_flag = False

# for i in data_lst:
#   print(len(i), i)

columns = ['CategoryNameoftheShareholders', 'NoOfShareholder', 'NoOfFullyPaidShares', 'Blank_1', 'Blank_2', 'TotalNoSharesHeld', 'ShareholdingPerc']
pandas_df = pd.DataFrame(data_lst, columns=columns)

# Replace null values and empty spaces with 0
# pandas_df.replace(to_replace=['', ' ', None, pd.NA], value=0, inplace=True)

# Convert data types
pandas_df['NoOfShareholder'] = pandas_df['NoOfShareholder'].astype(int)
pandas_df['ShareholdingPerc'] = pandas_df['ShareholdingPerc'].astype(float)

# Filter rows where NoOfShareholder is equal to 1
pandas_df = pandas_df[pandas_df['NoOfShareholder'] == 1]

# Drop the original Blank_1 and Blank_2 columns
pandas_df.drop(columns=['Blank_1', 'Blank_2', 'NoOfShareholder', 'TotalNoSharesHeld'], inplace=True)
pandas_df['Number'] = Number
pandas_df['Title'] = title
pandas_df['Qtr'] = Qtr
pandas_df['Year'] = Yer
pandas_df